# Module 4: Comparison and Evaluation

**Duration:** ~25 minutes

## Learning Objectives
- Compare responses from all three approaches
- Apply a structured evaluation rubric
- Visualize performance differences
- Understand when to use each approach

## Overview
Now we'll bring together the results from all three approaches (baseline, RAG, and fine-tuned) and evaluate them systematically.

## 4.1 Load All Results

In [ ]:
import json
import os

# Clone repo if running in Colab and results don't exist
if not os.path.exists('results'):
    if os.path.exists('llm-finetuning-rag-lab'):
        os.chdir('llm-finetuning-rag-lab')
    else:
        print("Cloning repository...")
        !git clone https://github.com/therealnoof/llm-finetuning-rag-lab.git
        os.chdir('llm-finetuning-rag-lab')
        print("✅ Repository cloned!")

print(f"Working directory: {os.getcwd()}")
print(f"\nResults files:")
!ls -la results/ 2>/dev/null || echo "No results directory yet - run previous modules first"

In [ ]:
# Load all results
def load_results(filepath):
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"⚠️  Not found: {filepath}")
        return None

baseline_data = load_results('results/baseline_results.json')
rag_data = load_results('results/rag_results.json')
finetuned_data = load_results('results/finetuned_results.json')

# Check what we have
print("\n📊 Results status:")

def status_line(name, data):
    if data:
        count = len(data.get('responses', []))
        return f"   {name}: ✅ Loaded ({count} responses)"
    else:
        return f"   {name}: ❌ Missing - will generate"

print(status_line("Baseline  ", baseline_data))
print(status_line("RAG       ", rag_data))
print(status_line("Fine-tuned", finetuned_data))

# Flag to track if we need to generate any results
needs_generation = not baseline_data or not rag_data or not finetuned_data
if needs_generation:
    print("\n⚠️  Some results are missing. Run the next cell to generate them.")
    print("   This will load TinyLlama and generate responses (takes a few minutes).")
else:
    print("\n✅ All results loaded! Skip the generation cell and proceed to comparisons.")

## 4.1b Generate Missing Results (Optional)

**Run this cell only if results are missing above.** This will:
1. Load TinyLlama with 4-bit quantization
2. Generate baseline responses (if missing)
3. Set up RAG and generate responses (if missing)
4. Load fine-tuned adapter and generate responses (if available)

⏱️ This takes approximately 5-10 minutes depending on GPU availability.

In [ ]:
# ============================================================================
# GENERATE MISSING RESULTS
# Skip this cell if all results were loaded above
# ============================================================================

if not needs_generation:
    print("✅ All results already loaded - skipping generation.")
else:
    print("🔧 Installing dependencies...")
    !pip install -q transformers accelerate bitsandbytes torch sentencepiece
    !pip install -q langchain langchain-community langchain-huggingface chromadb
    !pip install -q sentence-transformers langchain_text_splitters
    
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
    
    # Evaluation questions (same as used in other modules)
    EVAL_QUESTIONS = [
        "What is a virtual server in F5 BIG-IP and what is its purpose?",
        "How do I configure SSL offloading on F5 BIG-IP?",
        "Write an iRule to redirect HTTP to HTTPS.",
        "What is the difference between Least Connections and Round Robin load balancing?",
        "How do I troubleshoot a pool member that is marked as offline?"
    ]
    
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    model = None
    tokenizer = None
    text_gen_pipeline = None
    
    def load_base_model():
        """Load TinyLlama with 4-bit quantization."""
        global model, tokenizer, text_gen_pipeline
        if model is not None:
            return  # Already loaded
        
        print("\n📦 Loading TinyLlama with 4-bit quantization...")
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        text_gen_pipeline = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        print("✅ Model loaded!")
    
    def generate_response(question, context=None):
        """Generate a response using the loaded model."""
        if context:
            prompt = f"""<|system|>
You are an F5 BIG-IP technical expert. Use the following context to answer the question.

Context:
{context}</s>
<|user|>
{question}</s>
<|assistant|>
"""
        else:
            prompt = f"""<|system|>
You are a helpful technical assistant.</s>
<|user|>
{question}</s>
<|assistant|>
"""
        
        result = text_gen_pipeline(prompt)[0]["generated_text"]
        # Extract just the assistant's response
        if "<|assistant|>" in result:
            response = result.split("<|assistant|>")[-1].strip()
        else:
            response = result[len(prompt):].strip()
        return response
    
    # Ensure results directory exists
    os.makedirs('results', exist_ok=True)
    
    # ========== GENERATE BASELINE RESULTS ==========
    if not baseline_data:
        load_base_model()
        print("\n" + "="*60)
        print("GENERATING BASELINE RESPONSES")
        print("="*60)
        
        baseline_responses = []
        for i, question in enumerate(EVAL_QUESTIONS):
            print(f"\n[{i+1}/{len(EVAL_QUESTIONS)}] {question[:50]}...")
            response = generate_response(question)
            baseline_responses.append({
                "question": question,
                "response": response
            })
            print(f"   Response: {response[:100]}...")
        
        baseline_data = {
            "model": MODEL_NAME,
            "type": "baseline",
            "responses": baseline_responses
        }
        
        with open('results/baseline_results.json', 'w') as f:
            json.dump(baseline_data, f, indent=2)
        print("\n✅ Baseline results saved!")
    
    # ========== GENERATE RAG RESULTS ==========
    if not rag_data:
        load_base_model()
        print("\n" + "="*60)
        print("SETTING UP RAG AND GENERATING RESPONSES")
        print("="*60)
        
        from langchain_community.document_loaders import DirectoryLoader, TextLoader
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        from langchain_community.vectorstores import Chroma
        from langchain_huggingface import HuggingFaceEmbeddings
        
        # Load and chunk documents
        print("\n📄 Loading F5 documentation...")
        loader = DirectoryLoader('data/f5_docs/', glob="*.txt", loader_cls=TextLoader)
        documents = loader.load()
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(documents)
        print(f"   Created {len(chunks)} chunks from {len(documents)} documents")
        
        # Create vector store
        print("\n🔢 Creating embeddings...")
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectorstore = Chroma.from_documents(chunks, embeddings)
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        print("✅ Vector store created!")
        
        # Generate RAG responses
        print("\n🔍 Generating RAG responses...")
        rag_responses = []
        for i, question in enumerate(EVAL_QUESTIONS):
            print(f"\n[{i+1}/{len(EVAL_QUESTIONS)}] {question[:50]}...")
            
            # Retrieve relevant documents
            retrieved_docs = retriever.invoke(question)
            context = "\n\n".join(doc.page_content for doc in retrieved_docs)
            sources = list(set(doc.metadata.get('source', 'unknown').split('/')[-1] 
                              for doc in retrieved_docs))
            
            # Generate response with context
            response = generate_response(question, context=context)
            
            rag_responses.append({
                "question": question,
                "response": response,
                "sources": sources
            })
            print(f"   Sources: {sources}")
            print(f"   Response: {response[:100]}...")
        
        rag_data = {
            "model": MODEL_NAME,
            "type": "rag",
            "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
            "chunk_size": 500,
            "responses": rag_responses
        }
        
        with open('results/rag_results.json', 'w') as f:
            json.dump(rag_data, f, indent=2)
        print("\n✅ RAG results saved!")
        
        # Cleanup vectorstore
        del vectorstore, retriever
    
    # ========== GENERATE FINE-TUNED RESULTS ==========
    if not finetuned_data:
        print("\n" + "="*60)
        print("GENERATING FINE-TUNED RESPONSES")
        print("="*60)
        
        # Check if fine-tuned adapter exists
        adapter_path = "outputs/f5-expert-tinyllama"
        if os.path.exists(adapter_path):
            print(f"\n🎯 Loading fine-tuned adapter from {adapter_path}...")
            from peft import PeftModel
            
            # Load base model if not already loaded
            load_base_model()
            
            # Load adapter
            model = PeftModel.from_pretrained(model, adapter_path)
            text_gen_pipeline = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
            print("✅ Fine-tuned adapter loaded!")
            
            ft_responses = []
            for i, question in enumerate(EVAL_QUESTIONS):
                print(f"\n[{i+1}/{len(EVAL_QUESTIONS)}] {question[:50]}...")
                response = generate_response(question)
                ft_responses.append({
                    "question": question,
                    "response": response
                })
                print(f"   Response: {response[:100]}...")
            
            finetuned_data = {
                "model": MODEL_NAME,
                "type": "finetuned",
                "adapter": "f5-expert-lora",
                "responses": ft_responses
            }
            
            with open('results/finetuned_results.json', 'w') as f:
                json.dump(finetuned_data, f, indent=2)
            print("\n✅ Fine-tuned results saved!")
        else:
            print(f"\n⚠️  Fine-tuned adapter not found at {adapter_path}")
            print("   Run Module 3 (Fine-Tuning) first to create the adapter.")
            print("   Using baseline responses as placeholder for comparison...")
            
            # Create placeholder using baseline (marked clearly)
            if baseline_data:
                finetuned_data = {
                    "model": MODEL_NAME,
                    "type": "finetuned",
                    "adapter": "NOT_TRAINED - using baseline as placeholder",
                    "responses": baseline_data['responses'].copy()
                }
                print("   ⚠️  Fine-tuned results are placeholder only!")
    
    # Cleanup
    if model is not None:
        del model, tokenizer, text_gen_pipeline
        torch.cuda.empty_cache()
        print("\n🧹 GPU memory cleaned up!")
    
    print("\n" + "="*60)
    print("✅ GENERATION COMPLETE")
    print("="*60)
    print("\n📊 Final results status:")
    print(f"   Baseline:   {'✅' if baseline_data else '❌'}")
    print(f"   RAG:        {'✅' if rag_data else '❌'}")
    print(f"   Fine-tuned: {'✅' if finetuned_data else '❌'}")

## 4.2 Side-by-Side Comparison

Let's view responses from all three approaches for each question.

In [ ]:
def display_comparison(question_idx=0):
    """Display responses from all three approaches for a question."""
    print("=" * 80)
    print(f"QUESTION {question_idx + 1}:")
    print(baseline_data['responses'][question_idx]['question'])
    print("=" * 80)
    
    print("\n" + "─" * 40)
    print("📦 BASELINE RESPONSE:")
    print("─" * 40)
    print(baseline_data['responses'][question_idx]['response'])
    
    print("\n" + "─" * 40)
    print("🔍 RAG RESPONSE:")
    print("─" * 40)
    print(rag_data['responses'][question_idx]['response'])
    if 'sources' in rag_data['responses'][question_idx]:
        print(f"\nSources: {', '.join(rag_data['responses'][question_idx]['sources'])}")
    
    print("\n" + "─" * 40)
    print("🎯 FINE-TUNED RESPONSE:")
    print("─" * 40)
    print(finetuned_data['responses'][question_idx]['response'])

# Display first comparison
display_comparison(0)

In [ ]:
# Display all comparisons
for i in range(len(baseline_data['responses'])):
    display_comparison(i)
    print("\n")

## 4.3 Evaluation Rubric

We'll score each response on 5 criteria (0-5 scale each):

| Criterion | Description |
|-----------|-------------|
| **Accuracy** | Factual correctness of technical information |
| **Completeness** | How fully the question is answered |
| **Specificity** | Inclusion of F5-specific details (TMSH, iRules, etc.) |
| **Actionability** | Practical usefulness - can someone act on this? |
| **Clarity** | Organization and understandability |

### Scoring Guide

**0** - Completely wrong/missing  
**1** - Mostly incorrect  
**2** - Partially correct  
**3** - Reasonably good  
**4** - Good with minor issues  
**5** - Excellent

In [ ]:
# Create evaluation structure
evaluations = {
    'questions': [],
    'baseline_scores': [],
    'rag_scores': [],
    'finetuned_scores': []
}

criteria = ['accuracy', 'completeness', 'specificity', 'actionability', 'clarity']

# Initialize with example scores (you can modify these)
# In a real evaluation, these would be filled in manually or by a judge

example_scores = {
    'baseline': [
        {'accuracy': 2, 'completeness': 2, 'specificity': 1, 'actionability': 2, 'clarity': 3},
        {'accuracy': 2, 'completeness': 2, 'specificity': 1, 'actionability': 1, 'clarity': 3},
        {'accuracy': 1, 'completeness': 1, 'specificity': 0, 'actionability': 1, 'clarity': 2},
        {'accuracy': 3, 'completeness': 2, 'specificity': 1, 'actionability': 2, 'clarity': 3},
        {'accuracy': 2, 'completeness': 2, 'specificity': 1, 'actionability': 2, 'clarity': 3}
    ],
    'rag': [
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 3, 'specificity': 4, 'actionability': 4, 'clarity': 3},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 3, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4}
    ],
    'finetuned': [
        {'accuracy': 4, 'completeness': 4, 'specificity': 5, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 5, 'actionability': 5, 'clarity': 4},
        {'accuracy': 5, 'completeness': 4, 'specificity': 5, 'actionability': 5, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 5, 'actionability': 5, 'clarity': 4}
    ]
}

print("📝 Example evaluation scores loaded")
print("\nNote: In a real evaluation, you would score each response manually.")
print("The example scores demonstrate expected patterns.")

## 4.4 Calculate Summary Statistics

In [ ]:
import numpy as np

def calculate_stats(scores):
    """Calculate average scores for each criterion and overall."""
    stats = {criterion: [] for criterion in criteria}
    
    for score_set in scores:
        for criterion in criteria:
            stats[criterion].append(score_set[criterion])
    
    averages = {criterion: np.mean(values) for criterion, values in stats.items()}
    averages['total'] = sum(averages.values())
    averages['average'] = averages['total'] / len(criteria)
    
    return averages

baseline_stats = calculate_stats(example_scores['baseline'])
rag_stats = calculate_stats(example_scores['rag'])
finetuned_stats = calculate_stats(example_scores['finetuned'])

print("📊 Summary Statistics\n")
print(f"{'Criterion':<15} {'Baseline':>10} {'RAG':>10} {'Fine-tuned':>12}")
print("-" * 50)
for criterion in criteria:
    print(f"{criterion.title():<15} {baseline_stats[criterion]:>10.2f} {rag_stats[criterion]:>10.2f} {finetuned_stats[criterion]:>12.2f}")
print("-" * 50)
print(f"{'Average':<15} {baseline_stats['average']:>10.2f} {rag_stats['average']:>10.2f} {finetuned_stats['average']:>12.2f}")
print(f"{'Total (/25)':<15} {baseline_stats['total']:>10.2f} {rag_stats['total']:>10.2f} {finetuned_stats['total']:>12.2f}")

## 4.5 Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Enable inline plotting
%matplotlib inline

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Grouped bar chart by criteria
x = np.arange(len(criteria))
width = 0.25

ax1 = axes[0]
bars1 = ax1.bar(x - width, [baseline_stats[c] for c in criteria], width, label='Baseline', color='#ff6b6b')
bars2 = ax1.bar(x, [rag_stats[c] for c in criteria], width, label='RAG', color='#4ecdc4')
bars3 = ax1.bar(x + width, [finetuned_stats[c] for c in criteria], width, label='Fine-tuned', color='#45b7d1')

ax1.set_xlabel('Evaluation Criteria', fontsize=12)
ax1.set_ylabel('Score (0-5)', fontsize=12)
ax1.set_title('Performance by Criteria', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([c.title() for c in criteria], rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0, 5.5)
ax1.axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Threshold')

# Chart 2: Overall average comparison
ax2 = axes[1]
approaches = ['Baseline', 'RAG', 'Fine-tuned']
averages = [baseline_stats['average'], rag_stats['average'], finetuned_stats['average']]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']

bars = ax2.bar(approaches, averages, color=colors)

ax2.set_xlabel('Approach', fontsize=12)
ax2.set_ylabel('Average Score (0-5)', fontsize=12)
ax2.set_title('Overall Performance Comparison', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 5.5)
ax2.axhline(y=3, color='gray', linestyle='--', alpha=0.5)

# Add value labels on bars
for bar, val in zip(bars, averages):
    ax2.annotate(f'{val:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5),
                textcoords='offset points',
                ha='center', va='bottom',
                fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('results/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Chart saved to results/comparison_chart.png")

In [ ]:
# Radar chart for detailed comparison
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Prepare data
angles = np.linspace(0, 2 * np.pi, len(criteria), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

baseline_values = [baseline_stats[c] for c in criteria] + [baseline_stats[criteria[0]]]
rag_values = [rag_stats[c] for c in criteria] + [rag_stats[criteria[0]]]
finetuned_values = [finetuned_stats[c] for c in criteria] + [finetuned_stats[criteria[0]]]

# Plot
ax.plot(angles, baseline_values, 'o-', linewidth=2, label='Baseline', color='#ff6b6b')
ax.fill(angles, baseline_values, alpha=0.1, color='#ff6b6b')

ax.plot(angles, rag_values, 'o-', linewidth=2, label='RAG', color='#4ecdc4')
ax.fill(angles, rag_values, alpha=0.1, color='#4ecdc4')

ax.plot(angles, finetuned_values, 'o-', linewidth=2, label='Fine-tuned', color='#45b7d1')
ax.fill(angles, finetuned_values, alpha=0.1, color='#45b7d1')

# Customize
ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.title() for c in criteria], fontsize=11)
ax.set_ylim(0, 5)
ax.set_title('Multi-dimensional Performance Comparison', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('results/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Radar chart saved to results/radar_chart.png")

## 4.6 Analysis and Key Findings

In [ ]:
print("="*60)
print("KEY FINDINGS")
print("="*60)

# Calculate improvements
rag_improvement = ((rag_stats['average'] - baseline_stats['average']) / baseline_stats['average']) * 100
ft_improvement = ((finetuned_stats['average'] - baseline_stats['average']) / baseline_stats['average']) * 100

print(f"\n📈 Improvement over Baseline:")
print(f"   RAG: +{rag_improvement:.1f}%")
print(f"   Fine-tuned: +{ft_improvement:.1f}%")

# Find best approach per criterion
print(f"\n🏆 Best Approach by Criterion:")
for criterion in criteria:
    scores = {
        'Baseline': baseline_stats[criterion],
        'RAG': rag_stats[criterion],
        'Fine-tuned': finetuned_stats[criterion]
    }
    best = max(scores, key=scores.get)
    print(f"   {criterion.title()}: {best} ({scores[best]:.2f})")

# Overall winner
overall_scores = {
    'Baseline': baseline_stats['average'],
    'RAG': rag_stats['average'],
    'Fine-tuned': finetuned_stats['average']
}
winner = max(overall_scores, key=overall_scores.get)
print(f"\n🥇 Overall Best: {winner} (avg: {overall_scores[winner]:.2f}/5)")

## 4.7 When to Use Each Approach

Based on our evaluation, here are recommendations for when to use each approach:

In [ ]:
recommendations = """
╔══════════════════════════════════════════════════════════════════╗
║                    APPROACH RECOMMENDATIONS                       ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  📦 BASELINE (General LLM)                                       ║
║  ─────────────────────────────                                   ║
║  ✓ General knowledge questions                                   ║
║  ✓ Quick prototyping                                             ║
║  ✓ When no domain data available                                 ║
║  ✗ Poor for domain-specific details                              ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🔍 RAG (Retrieval-Augmented Generation)                         ║
║  ─────────────────────────────────────────                       ║
║  ✓ Factual queries needing source verification                   ║
║  ✓ Frequently updated knowledge bases                            ║
║  ✓ When source attribution is important                          ║
║  ✓ Limited compute/training resources                            ║
║  ✗ Depends on retrieval quality                                  ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🎯 FINE-TUNED                                                   ║
║  ──────────────                                                  ║
║  ✓ Consistent domain terminology needed                          ║
║  ✓ Reasoning and pattern-based questions                         ║
║  ✓ Fast inference without retrieval latency                      ║
║  ✓ When training data is available                               ║
║  ✗ Harder to update; risk of overfitting                         ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🚀 PRODUCTION RECOMMENDATION: COMBINE APPROACHES                ║
║  ───────────────────────────────────────────────                 ║
║  • Fine-tune for domain understanding and terminology            ║
║  • Add RAG for current/detailed reference information            ║
║  • Implement fallback strategies for edge cases                  ║
║  • Monitor and continuously evaluate performance                 ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""

print(recommendations)

## 4.8 Save Evaluation Report

In [ ]:
from datetime import datetime

# Compile full evaluation report
report = {
    "timestamp": datetime.now().isoformat(),
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "summary": {
        "baseline": {
            "average_score": baseline_stats['average'],
            "total_score": baseline_stats['total'],
            "by_criterion": {c: baseline_stats[c] for c in criteria}
        },
        "rag": {
            "average_score": rag_stats['average'],
            "total_score": rag_stats['total'],
            "by_criterion": {c: rag_stats[c] for c in criteria},
            "improvement_over_baseline": rag_improvement
        },
        "finetuned": {
            "average_score": finetuned_stats['average'],
            "total_score": finetuned_stats['total'],
            "by_criterion": {c: finetuned_stats[c] for c in criteria},
            "improvement_over_baseline": ft_improvement
        }
    },
    "winner": winner,
    "detailed_scores": example_scores,
    "questions": [r['question'] for r in baseline_data['responses']]
}

with open('results/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✅ Evaluation report saved to results/evaluation_report.json")

In [ ]:
# Generate text report
text_report = f"""
================================================================================
F5 AI TECHNICAL ASSISTANT - EVALUATION REPORT
================================================================================

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Base Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0

--------------------------------------------------------------------------------
SUMMARY SCORES (Scale: 0-5)
--------------------------------------------------------------------------------

Approach        Accuracy  Completeness  Specificity  Actionability  Clarity  AVG
--------------------------------------------------------------------------------
Baseline        {baseline_stats['accuracy']:.2f}      {baseline_stats['completeness']:.2f}          {baseline_stats['specificity']:.2f}          {baseline_stats['actionability']:.2f}          {baseline_stats['clarity']:.2f}     {baseline_stats['average']:.2f}
RAG             {rag_stats['accuracy']:.2f}      {rag_stats['completeness']:.2f}          {rag_stats['specificity']:.2f}          {rag_stats['actionability']:.2f}          {rag_stats['clarity']:.2f}     {rag_stats['average']:.2f}
Fine-tuned      {finetuned_stats['accuracy']:.2f}      {finetuned_stats['completeness']:.2f}          {finetuned_stats['specificity']:.2f}          {finetuned_stats['actionability']:.2f}          {finetuned_stats['clarity']:.2f}     {finetuned_stats['average']:.2f}

--------------------------------------------------------------------------------
IMPROVEMENT OVER BASELINE
--------------------------------------------------------------------------------

RAG:        +{rag_improvement:.1f}%
Fine-tuned: +{ft_improvement:.1f}%

--------------------------------------------------------------------------------
OVERALL WINNER: {winner.upper()}
--------------------------------------------------------------------------------

KEY TAKEAWAYS:
• Both RAG and fine-tuning significantly improve F5-specific responses
• Fine-tuning excels at terminology and actionable guidance
• RAG provides reliable source-backed information
• For production: combine both approaches for best results

================================================================================
"""

with open('results/evaluation_report.txt', 'w') as f:
    f.write(text_report)

print(text_report)
print("✅ Text report saved to results/evaluation_report.txt")

## 4.9 Summary

In this module, we:

1. ✅ Loaded and compared all three approaches
2. ✅ Applied a structured evaluation rubric
3. ✅ Created visualizations of performance
4. ✅ Analyzed strengths and weaknesses
5. ✅ Generated comprehensive reports

### Key Takeaways

1. **Baseline limitations**: General LLMs lack domain expertise
2. **RAG benefits**: Brings external knowledge without retraining
3. **Fine-tuning benefits**: Internalizes domain patterns and terminology
4. **Best practice**: Combine approaches for production systems

### What's Next?

For production deployment, consider:
- Expanding training data (more Q&A pairs)
- Adding more documentation to RAG
- Implementing hybrid RAG + fine-tuned approach
- Setting up monitoring and evaluation pipelines
- Testing with real users and collecting feedback

## 🎉 Congratulations!

You've completed the F5 AI Technical Assistant Training Lab!

You've learned how to:
- Load and use quantized LLMs
- Build RAG systems with LangChain and ChromaDB
- Fine-tune models with QLoRA and Unsloth
- Evaluate and compare different approaches

These skills apply broadly to building domain-specific AI assistants for any technical field!